In [ ]:

!pip install -q "numpy<2" scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 5.7 MB/s eta 0:00:00


In [ ]:
# Cell 1.5: Restart the runtime after downgrading numpy
import os
os.kill(os.getpid(), 9)

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile, urllib.request, os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import cross_val_score

from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate, train_test_split
from surprise import accuracy

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported!")
print("NumPy version:", np.__version__)   # Should show 1.x.x

In [ ]:

url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
urllib.request.urlretrieve(url, "ml-100k.zip")

with zipfile.ZipFile("ml-100k.zip", "r") as zip_ref:
    zip_ref.extractall(".")

print(" Dataset downloaded and extracted!")

In [ ]:
# Cell 4: Load movies and ratings data
ratings = pd.read_csv('ml-100k/u.data', sep='\t',
                      names=['user_id', 'movie_id', 'rating', 'timestamp'])

movies = pd.read_csv('ml-100k/u.item', sep='|', encoding='latin-1',
                     names=['movie_id','title','release_date','video_release',
                            'IMDb_URL','unknown','Action','Adventure','Animation',
                            'Children','Comedy','Crime','Documentary','Drama',
                            'Fantasy','Film-Noir','Horror','Musical','Mystery',
                            'Romance','Sci-Fi','Thriller','War','Western'])

print("🎬 Movies shape:", movies.shape)
print("⭐ Ratings shape:", ratings.shape)
print("\n--- Sample Movies ---")
display(movies[['movie_id','title']].head())
print("\n--- Sample Ratings ---")
display(ratings.head())

In [ ]:
# Cell 5: EDA - Rating distribution
plt.figure(figsize=(14, 4))

# Plot 1: Rating distribution
plt.subplot(1, 3, 1)
ratings['rating'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')

# Plot 2: Ratings per user
plt.subplot(1, 3, 2)
ratings_per_user = ratings.groupby('user_id')['rating'].count()
plt.hist(ratings_per_user, bins=40, color='salmon', edgecolor='black')
plt.title('Ratings per User')
plt.xlabel('Number of Ratings')
plt.ylabel('Users')

# Plot 3: Ratings per movie
plt.subplot(1, 3, 3)
ratings_per_movie = ratings.groupby('movie_id')['rating'].count()
plt.hist(ratings_per_movie, bins=40, color='mediumseagreen', edgecolor='black')
plt.title('Ratings per Movie')
plt.xlabel('Number of Ratings')
plt.ylabel('Movies')

plt.tight_layout()
plt.show()
print(f"Total users: {ratings['user_id'].nunique()} | Total movies: {ratings['movie_id'].nunique()} | Total ratings: {len(ratings)}")

In [ ]:
# Cell 6: EDA - Top rated movies & genre popularity
genre_cols = ['Action','Adventure','Animation','Children','Comedy','Crime',
              'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical',
              'Mystery','Romance','Sci-Fi','Thriller','War','Western']

plt.figure(figsize=(14, 5))

# Plot 1: Top 10 most rated movies
plt.subplot(1, 2, 1)
top_movies = ratings.groupby('movie_id')['rating'].count().reset_index()
top_movies = top_movies.merge(movies[['movie_id','title']], on='movie_id')
top_movies = top_movies.sort_values('rating', ascending=False).head(10)
sns.barplot(x='rating', y='title', data=top_movies, palette='Blues_d')
plt.title('Top 10 Most Rated Movies')
plt.xlabel('Number of Ratings')
plt.ylabel('')

# Plot 2: Genre popularity
plt.subplot(1, 2, 2)
genre_counts = movies[genre_cols].sum().sort_values(ascending=False)
sns.barplot(x=genre_counts.values, y=genre_counts.index, palette='Oranges_d')
plt.title('Genre Popularity')
plt.xlabel('Number of Movies')
plt.ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Content-Based Filtering using TF-IDF on genres

# Build genre string for each movie
movies['genres'] = movies[genre_cols].apply(
    lambda row: ' '.join([genre for genre, val in zip(genre_cols, row) if val == 1]), axis=1
)

# TF-IDF Vectorizer on genres
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])

# Cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Index mapping: title -> index
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def content_based_recommend(title, n=10):
    """Get top-n movie recommendations based on content similarity."""
    if title not in indices:
        return f" Movie '{title}' not found in dataset."

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n+1]
    movie_indices = [i[0] for i in sim_scores]
    scores = [round(i[1], 4) for i in sim_scores]

    result = movies[['title']].iloc[movie_indices].copy()
    result['similarity_score'] = scores
    result.reset_index(drop=True, inplace=True)
    result.index += 1
    return result

# Test it!
print("Content-Based Recommendations for 'Toy Story (1995)':\n")
display(content_based_recommend('Toy Story (1995)'))

In [ ]:
# Cell 8: Collaborative Filtering using SVD (Surprise library)

# Prepare data for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['user_id', 'movie_id', 'rating']], reader)

# Train/Test split
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Train SVD model
svd_model = SVD(n_factors=100, random_state=42)
svd_model.fit(trainset)

# Evaluate
predictions = svd_model.test(testset)
rmse = accuracy.rmse(predictions)
mae  = accuracy.mae(predictions)

print(f"\n✅ SVD Model Trained!")
print(f"📉 RMSE: {rmse:.4f}")
print(f"📉 MAE : {mae:.4f}")

def collab_recommend(user_id, n=10):
    """Recommend top-n movies for a given user using SVD."""
    rated_movies = ratings[ratings['user_id'] == user_id]['movie_id'].tolist()
    all_movie_ids = movies['movie_id'].tolist()
    unrated = [m for m in all_movie_ids if m not in rated_movies]

    preds = [(mid, svd_model.predict(user_id, mid).est) for mid in unrated]
    preds_sorted = sorted(preds, key=lambda x: x[1], reverse=True)[:n]

    rec_ids = [p[0] for p in preds_sorted]
    rec_scores = [round(p[1], 4) for p in preds_sorted]

    result = movies[movies['movie_id'].isin(rec_ids)][['movie_id','title']].copy()
    score_map = dict(zip(rec_ids, rec_scores))
    result['predicted_rating'] = result['movie_id'].map(score_map)
    result = result.sort_values('predicted_rating', ascending=False)
    result.reset_index(drop=True, inplace=True)
    result.index += 1
    return result

# Test for user 1
print("\n🎬 Collaborative Filtering Recommendations for User 1:\n")
display(collab_recommend(user_id=1))

In [ ]:
# Cell 9: Compare both models visually

# Cross-validate SVD for robust metrics
cv_results = cross_validate(svd_model, data, measures=['RMSE', 'MAE'], cv=5, verbose=False)

svd_rmse = cv_results['test_rmse'].mean()
svd_mae  = cv_results['test_mae'].mean()

# Content-based: measure avg similarity score as proxy metric
sample_movies = ['Toy Story (1995)', 'GoodFellas (1990)', 'Fargo (1996)']
cb_scores = []
for m in sample_movies:
    recs = content_based_recommend(m)
    if isinstance(recs, pd.DataFrame):
        cb_scores.append(recs['similarity_score'].mean())
avg_cb_similarity = np.mean(cb_scores)

# --- Comparison Table ---
print("=" * 50)
print("        📊 MODEL COMPARISON SUMMARY")
print("=" * 50)
print(f"  Content-Based Filtering")
print(f"    Avg Similarity Score : {avg_cb_similarity:.4f}")
print(f"    Strength             : Genre/tag similarity")
print(f"    Weakness             : No personalization")
print()
print(f"  Collaborative Filtering (SVD)")
print(f"    RMSE (5-fold CV)     : {svd_rmse:.4f}")
print(f"    MAE  (5-fold CV)     : {svd_mae:.4f}")
print(f"    Strength             : Personalized for users")
print(f"    Weakness             : Cold-start problem")
print("=" * 50)

# --- Bar Chart Comparison ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: SVD CV metrics
metrics = ['RMSE', 'MAE']
values  = [svd_rmse, svd_mae]
axes[0].bar(metrics, values, color=['steelblue', 'salmon'], edgecolor='black', width=0.4)
axes[0].set_title('Collaborative Filtering (SVD)\n5-Fold CV Metrics')
axes[0].set_ylabel('Error Score (lower is better)')
axes[0].set_ylim(0, 1.5)
for i, v in enumerate(values):
    axes[0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

# Plot 2: Content-based similarity per movie
axes[1].bar(sample_movies, cb_scores, color='mediumseagreen', edgecolor='black')
axes[1].set_title('Content-Based Filtering\nAvg Similarity Score per Seed Movie')
axes[1].set_ylabel('Cosine Similarity (higher is better)')
axes[1].set_ylim(0, 1)
axes[1].set_xticklabels(sample_movies, rotation=15, ha='right')
for i, v in enumerate(cb_scores):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Try it yourself!

print("=" * 55)
print("     🎬 MOVIE RECOMMENDATION SYSTEM - TRY IT!")
print("=" * 55)

# --- Content-Based ---
movie_name = "Star Wars (1977)"   # change this to any movie!
print(f"\n📽️  Content-Based → Similar to '{movie_name}':")
display(content_based_recommend(movie_name, n=5))

# --- Collaborative ---
user = 42    #  Change this to any user_id (1–943)!
print(f"\n👤 Collaborative (SVD) → Top picks for User {user}:")
display(collab_recommend(user_id=user, n=5))

print("\n✅ Done! Change movie_name or user above and re-run to explore.")